# H1B Visa Data Cleaning and Feature Selection

## Step 1 — Import Libraries

In [1]:
import pandas as pd

## Step 2 — Load Raw Files 
### Why?
We reload all quarterly datasets inside this notebook so the cleaning process is self-contained and reproducible.

In [2]:
q1 = pd.read_excel("../data/raw/LCA_Disclosure_Data_FY2024_Q1.xlsx")
q2 = pd.read_excel("../data/raw/LCA_Disclosure_Data_FY2024_Q2.xlsx")
q3 = pd.read_excel("../data/raw/LCA_Disclosure_Data_FY2024_Q3.xlsx")
q4 = pd.read_excel("../data/raw/LCA_Disclosure_Data_FY2024_Q4.xlsx")

## Step 3 — Select Relevant Columns
### Why?
We keep only the columns needed to answer project questions such as top employers, salaries, locations, job demand, and H1B strategy insights.

In [4]:
relevant_cols = [
    'CASE_NUMBER',
    'CASE_STATUS',
    'RECEIVED_DATE',
    'DECISION_DATE',
    'VISA_CLASS',
    'JOB_TITLE',
    'SOC_CODE',
    'SOC_TITLE',
    'FULL_TIME_POSITION',
    'BEGIN_DATE',
    'END_DATE',
    'TOTAL_WORKER_POSITIONS',
    'EMPLOYER_NAME',
    'EMPLOYER_CITY',
    'EMPLOYER_STATE',
    'NAICS_CODE',
    'WORKSITE_CITY',
    'WORKSITE_STATE',
    'WORKSITE_POSTAL_CODE',
    'WAGE_RATE_OF_PAY_FROM',
    'WAGE_RATE_OF_PAY_TO',
    'WAGE_UNIT_OF_PAY',
    'PREVAILING_WAGE',
    'PW_WAGE_LEVEL'
]
q1 = q1[relevant_cols]
q2 = q2[relevant_cols]
q3 = q3[relevant_cols]
q4 = q4[relevant_cols]

## Step 4 — Add Quarter Column
### Why?
Adding the quarter lets us compare filing trends across Q1, Q2, Q3, and Q4.

In [5]:
q1["quarter"] = "Q1"
q2["quarter"] = "Q2"
q3["quarter"] = "Q3"
q4["quarter"] = "Q4"

## Step 5 — Merge All Quarters
### Why?
Combining all quarters creates one complete FY2024 dataset for analysis.

In [6]:
df = pd.concat([q1, q2, q3, q4], ignore_index=True)
print("Merged Shape:", df.shape)

Merged Shape: (561037, 25)


## Step 6 — Basic Cleaning
### Why?
We remove duplicates, standardize text values, and prepare clean data for SQL queries.

In [7]:
df.drop_duplicates(inplace=True)

text_cols = [
    "CASE_STATUS","VISA_CLASS","JOB_TITLE","SOC_TITLE",
    "EMPLOYER_NAME","EMPLOYER_CITY","EMPLOYER_STATE",
    "WORKSITE_CITY","WORKSITE_STATE","PW_WAGE_LEVEL"
]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.upper()

## Step 7 — Convert Numeric Columns
### Why?
Salary and worker count columns must be numeric for calculations and charts.

In [8]:
num_cols = [
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_RATE_OF_PAY_TO",
    "PREVAILING_WAGE",
    "TOTAL_WORKER_POSITIONS"
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

## Step 8 — Convert Date Columns
### Why?
Date fields are converted into datetime format so we can calculate processing times, timelines, and trends across the year.

In [9]:
df['RECEIVED_DATE'] = pd.to_datetime(df['RECEIVED_DATE'], errors='coerce')
df['DECISION_DATE'] = pd.to_datetime(df['DECISION_DATE'], errors='coerce')
df['BEGIN_DATE']  = pd.to_datetime(df['BEGIN_DATE'], errors='coerce')
df['END_DATE']  = pd.to_datetime(df['END_DATE'], errors='coerce')

print("Date types fixed!")
print(df[['RECEIVED_DATE','DECISION_DATE','BEGIN_DATE','END_DATE']].dtypes)

Date types fixed!
RECEIVED_DATE    datetime64[ns]
DECISION_DATE    datetime64[ns]
BEGIN_DATE       datetime64[ns]
END_DATE         datetime64[ns]
dtype: object


## Step 9 — Create Standardized Annual Salary
### Why?
The raw dataset contains wages in different units such as Hour, Week, Month, and Year.  
To compare salaries fairly, we convert all wages into one common metric: annual salary.

In [10]:
# Fix suspicious hourly entries
df.loc[
    (df['WAGE_UNIT_OF_PAY'] == 'Hour') &
    (df['WAGE_RATE_OF_PAY_FROM'] > 500),
    'WAGE_UNIT_OF_PAY'
] = 'Year'

# Fix suspicious monthly entries
df.loc[
    (df['WAGE_UNIT_OF_PAY'] == 'Month') &
    (df['WAGE_RATE_OF_PAY_FROM'] > 50000),
    'WAGE_UNIT_OF_PAY'
] = 'Year'

# Convert all pay units to annual salary
def convert_to_annual(row):
    wage = row['WAGE_RATE_OF_PAY_FROM']
    unit = row['WAGE_UNIT_OF_PAY']

    if pd.isna(wage):
        return None
    elif unit == 'Hour':
        return wage * 40 * 52
    elif unit == 'Week':
        return wage * 52
    elif unit == 'Bi-Weekly':
        return wage * 26
    elif unit == 'Month':
        return wage * 12
    else:
        return wage

df['ANNUAL_SALARY'] = df.apply(convert_to_annual, axis=1)

## Step 10 — Remove Unrealistic Salary Values
### Why?
Very low or extremely high salaries are usually data-entry errors or test records. Removing them improves analysis quality.

In [11]:
before_rows = df.shape[0]

df = df[
    (df['ANNUAL_SALARY'] >= 30000) &
    (df['ANNUAL_SALARY'] <= 500000)
]

after_rows = df.shape[0]

print("Before:", before_rows)
print("After:", after_rows)
print("Removed:", before_rows - after_rows)

print("\n Final Salary Stats")
print(df['ANNUAL_SALARY'].describe())

Before: 561037
After: 559234
Removed: 1803

 Final Salary Stats
count    559234.000000
mean     123646.911501
std       52226.111074
min       30000.000000
25%       86965.000000
50%      112900.000000
75%      149781.000000
max      500000.000000
Name: ANNUAL_SALARY, dtype: float64


## Step 11 — Save Clean Dataset
### Why?
We export the final cleaned dataset to the processed folder for MySQL import and dashboarding.

In [13]:
df.to_csv("../data/processed/h1b_clean_2024.csv", index=False)
print("Clean file saved successfully.")

Clean file saved successfully.


## Step 12 — Final Check
### Why?
We verify the final structure before loading into SQL.

In [12]:
print(df.shape)
print(df.isnull().sum().sort_values(ascending=False).head(10))
df.head()
df.info()

(559234, 26)
WAGE_RATE_OF_PAY_TO    379101
CASE_NUMBER                 0
RECEIVED_DATE               0
DECISION_DATE               0
VISA_CLASS                  0
JOB_TITLE                   0
SOC_CODE                    0
SOC_TITLE                   0
FULL_TIME_POSITION          0
CASE_STATUS                 0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
Index: 559234 entries, 0 to 561036
Data columns (total 26 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   CASE_NUMBER             559234 non-null  object        
 1   CASE_STATUS             559234 non-null  object        
 2   RECEIVED_DATE           559234 non-null  datetime64[ns]
 3   DECISION_DATE           559234 non-null  datetime64[ns]
 4   VISA_CLASS              559234 non-null  object        
 5   JOB_TITLE               559234 non-null  object        
 6   SOC_CODE                559234 non-null  object        
 7   SOC_TITLE     